# Textual Data Representation for Biostatistics

This notebook walks through the main families of methods used to turn text into numbers
that statistical and machine-learning models can consume. We focus on a biomedical theme
(PubMed-style sentences, drug/disease names) so the running examples are familiar.

We will cover:

1. Classical sparse representations: Bag-of-Words and TF-IDF
2. Static word embeddings: Word2Vec
3. Sentence embeddings via Transformer encoders (sentence-transformers)
4. Modern large language model encoders accessed through the OpenRouter API (free models)

Each section includes a short conceptual note and a small hands-on example.


## 0. Setup

Install the libraries we need. Run this cell once.

In [ ]:
# Install dependencies (uncomment the line if running for the first time)
!pip install numpy scikit-learn matplotlib gensim sentence-transformers requests

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

np.random.seed(0)

### A small biomedical corpus

Throughout the notebook we will reuse this toy corpus of biomedical sentences.
It is intentionally tiny so cells run fast, but the methods scale to real PubMed
abstracts or clinical notes.

In [ ]:
corpus = [
    "Aspirin reduces the risk of myocardial infarction in high-risk patients.",
    "Ibuprofen is a non-steroidal anti-inflammatory drug used to relieve pain.",
    "Metformin is the first-line treatment for type 2 diabetes mellitus.",
    "Insulin therapy is required in many patients with type 1 diabetes.",
    "Hypertension increases the risk of stroke and cardiovascular disease.",
    "Smoking is a major risk factor for lung cancer and chronic obstructive pulmonary disease.",
    "Statins lower LDL cholesterol and reduce cardiovascular mortality.",
    "Obesity is associated with type 2 diabetes and metabolic syndrome.",
    "A randomized controlled trial showed that the new drug improved survival.",
    "The Cox proportional hazards model is widely used in survival analysis.",
    "Kaplan-Meier curves estimate the survival function from censored data.",
    "Logistic regression models the probability of a binary outcome.",
    "Bayesian methods incorporate prior information into the analysis.",
    "Confidence intervals quantify the uncertainty of an estimate.",
    "A p-value below 0.05 is conventionally considered statistically significant.",
    "Causal inference requires careful handling of confounding variables.",
    "Propensity score matching is used to balance covariates in observational studies.",
    "Meta-analysis combines results from multiple independent studies.",
    "Electronic health records are a rich source of real-world clinical data.",
    "Patients with diabetes often present with elevated blood glucose levels.",
]

print(f"Corpus size: {len(corpus)} sentences")
print("Example:", corpus[0])

## 1. Classical sparse representations

The simplest way to represent text is to count words. Two common variants:

- **Bag-of-Words (BoW)**: each document becomes a vector of raw word counts.
- **TF-IDF**: counts are reweighted so words that are frequent in a document but rare in the
  collection get higher weights. This downweights stopwords like "the" or "and".

Both produce a sparse, high-dimensional matrix of shape `(n_documents, vocabulary_size)`.
They ignore word order and semantics, but they are fast, interpretable, and surprisingly
competitive for many classification tasks.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Bag of Words
bow = CountVectorizer(stop_words="english")
X_bow = bow.fit_transform(corpus)
print("BoW matrix shape:", X_bow.shape)
print("Vocabulary size:", len(bow.vocabulary_))
print("First 15 vocabulary terms:", list(bow.vocabulary_.keys())[:15])

In [ ]:
import pandas as pd

# Convert the sparse matrix to a dense DataFrame with words as columns
df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=bow.get_feature_names_out()
)

# Show the first 5 documents and first 12 vocabulary columns
df_bow.iloc[:5, :12]


In [ ]:
# TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf.fit_transform(corpus)

# Inspect the top-weighted words in a single document
doc_idx = 2  # the metformin sentence
feature_names = np.array(tfidf.get_feature_names_out())

# X_tfidf[doc_idx] is a 1 x V sparse row (V = vocabulary size).
# .toarray() converts it to a dense 2D numpy array of shape (1, V).
# .ravel() flattens it to a 1D array of shape (V,), one TF-IDF weight per vocab word.
row = X_tfidf[doc_idx].toarray().ravel()

# np.argsort(row) returns the indices that would sort `row` in ASCENDING order.
# [::-1] reverses them, so now indices are sorted by TF-IDF weight DESCENDING.
# [:6] keeps the top 6 -- the six vocabulary indices with the highest weights
# in this document.
top = np.argsort(row)[::-1][:6]

print("Document:", corpus[doc_idx])
print("Top TF-IDF terms:")
for i in top:
    print(f"  {feature_names[i]:>20s}  {row[i]:.3f}")

### Document similarity with TF-IDF

We can compute cosine similarity between documents using their TF-IDF vectors.
Note how diabetes-related sentences cluster together even though the vectors only
encode word overlap.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(X_tfidf)

query_idx = 2  # "Metformin is the first-line treatment for type 2 diabetes mellitus."
ranked = np.argsort(sim[query_idx])[::-1]

print(f"Query: {corpus[query_idx]}\n")
print("Most similar documents:")
for i in ranked[1:5]:
    print(f"  ({sim[query_idx, i]:.2f})  {corpus[i]}")

**Limitation.** TF-IDF treats "diabetes" and "diabetic" as unrelated tokens, and it
cannot tell that "myocardial infarction" and "heart attack" mean the same thing.
We need representations that capture meaning, not just surface form.

## 2. Word2Vec: dense word embeddings

Word2Vec (Mikolov et al., 2013) learns a dense vector for each word such that words
appearing in similar contexts end up with similar vectors. Two training objectives:

- **CBOW**: predict a word given its surrounding context.
- **Skip-gram**: predict the surrounding context given a word.

The result is a vector space where semantic relationships become geometric:
`vector("king") - vector("man") + vector("woman")` lands near `vector("queen")`.

We will (a) train a tiny model on our corpus to see the API, and (b) load a real
pretrained model to see meaningful similarities.

In [ ]:
from gensim.models import Word2Vec
import re

def simple_tokenize(text):
    return re.findall(r"[a-zA-Z]+", text.lower())

tokenized_corpus = [simple_tokenize(doc) for doc in corpus]
print("Example tokenization:", tokenized_corpus[0])

# Train a tiny Word2Vec model. With only 20 sentences this will not produce great
# embeddings, but it shows the workflow.
w2v = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=5,
    min_count=1,
    sg=1,           # 1 = skip-gram, 0 = CBOW
    workers=2,
    epochs=50,
    seed=0,
)

print("\nVocabulary size:", len(w2v.wv))
print("Vector for 'diabetes' (first 8 dims):", w2v.wv["diabetes"][:8])

### Using a pretrained model

For real similarity queries we need a model trained on a large corpus. Gensim ships
a downloader that gives us access to several pretrained models. We use a small GloVe
model (~66 MB) trained on Wikipedia.

In [ ]:
import gensim.downloader as api

# Download a small pretrained model. First call will take a minute.
# Other options: 'word2vec-google-news-300' (1.6 GB), 'glove-twitter-25', etc.
pretrained = api.load("glove-wiki-gigaword-50")

print("Pretrained vocabulary size:", len(pretrained))
print("\nWords most similar to 'diabetes':")
for word, score in pretrained.most_similar("diabetes", topn=8):
    print(f"  {word:>15s}  {score:.3f}")

In [ ]:
# The classic analogy: king - man + woman = ?
result = pretrained.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print("king - man + woman =")
for word, score in result:
    print(f"  {word:>10s}  {score:.3f}")

# A biomedical analogy: diabetes is to insulin as hypertension is to ?
result = pretrained.most_similar(
    positive=["hypertension", "insulin"], negative=["diabetes"], topn=5
)
print("\nhypertension + insulin - diabetes =")
for word, score in result:
    print(f"  {word:>15s}  {score:.3f}")

### Visualizing the embedding space

We can project high-dimensional embeddings down to 2D with PCA to get an
intuitive picture. Notice how related concepts (diseases, drug classes, statistical
methods) form rough clusters.

In [ ]:
from sklearn.decomposition import PCA

words = [
    "diabetes", "insulin", "glucose", "metformin",
    "hypertension", "stroke", "cholesterol", "statin",
    "cancer", "tumor", "chemotherapy", "radiation",
    "regression", "variance", "probability", "estimate",
]
vectors = np.array([pretrained[w] for w in words])

coords = PCA(n_components=2, random_state=0).fit_transform(vectors)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=40, color="steelblue")
for (x, y), w in zip(coords, words):
    plt.annotate(w, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=10)
plt.title("Word2Vec / GloVe embeddings projected to 2D (PCA)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Limitations of static embeddings.**

- One vector per word, no matter the context. "Cell" in "cancer cell" and "prison cell"
  share the same vector.
- Out-of-vocabulary words have no representation.
- To get a sentence vector you have to average word vectors, which loses syntactic
  structure.

The next generation of models fixes these issues by producing *contextual* embeddings.

## 3. Sentence embeddings with Transformer encoders

Modern encoders like BERT, RoBERTa and their many descendants are Transformer
networks pretrained on large text corpora with masked-language-model objectives.
They produce a different vector for the same word depending on its context.

The `sentence-transformers` library wraps these models so that one call gives you
a single fixed-size vector for an entire sentence or paragraph. We use
`all-MiniLM-L6-v2`, a 22-million-parameter model that returns 384-dimensional
vectors. For biomedical work, drop-in replacements include
`pritamdeka/S-PubMedBert-MS-MARCO` and `cambridgeltl/SapBERT-from-PubMedBERT-fulltext`.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(corpus, normalize_embeddings=True)

print("Embedding matrix shape:", embeddings.shape)
print("Each sentence is a vector of length", embeddings.shape[1])

### Semantic search

Because the embeddings encode meaning rather than surface form, we can match a
query against the corpus even when the wording differs.

In [ ]:
def search(query, top_k=5):
    q_vec = model.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q_vec  # cosine similarity since vectors are normalized
    order = np.argsort(scores)[::-1][:top_k]
    print(f"Query: {query}\n")
    for i in order:
        print(f"  ({scores[i]:.3f})  {corpus[i]}")

search("medication for high blood sugar")

In [ ]:
search("statistical method for time-to-event data")

### Clustering biomedical sentences

With sentence embeddings, clustering algorithms like K-means group documents by
*topic* rather than by word overlap.

In [ ]:
from sklearn.cluster import KMeans

k = 3
km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(embeddings)

for c in range(k):
    print(f"\nCluster {c}:")
    for i, lab in enumerate(km.labels_):
        if lab == c:
            print(f"  - {corpus[i]}")

### Comparing representations on the same task

To get a feel for how much representation matters, compute the similarity of two
paraphrases under TF-IDF and under sentence embeddings.

In [ ]:
s1 = "Aspirin lowers the probability of a heart attack."
s2 = "Acetylsalicylic acid reduces the risk of myocardial infarction."

# TF-IDF similarity
tfidf2 = TfidfVectorizer(stop_words="english").fit([s1, s2])
v1, v2 = tfidf2.transform([s1, s2]).toarray()
tfidf_sim = float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-12))

# Sentence-transformer similarity
e1, e2 = model.encode([s1, s2], normalize_embeddings=True)
emb_sim = float(np.dot(e1, e2))

print(f"TF-IDF cosine similarity cosine similarity:            {tfidf_sim:.3f}")
print(f"Sentence-embedding cosine similarity: {emb_sim:.3f}")

## 4. Modern LLM encoders via OpenRouter

Large language models (Llama, Mistral, Qwen, DeepSeek, Gemma, etc.) are typically
deployed behind APIs. **OpenRouter** is a gateway that exposes hundreds of models
through a single OpenAI-compatible interface. A subset of models is available at
zero cost (they end with the `:free` suffix); they have rate limits (typically
20 requests per minute) but no billing.

We will use a free chat model for two text-representation tasks that LLMs handle well:

1. **Zero-shot classification** of biomedical sentences.
2. **Pairwise similarity judgment** as an alternative to cosine similarity on embeddings.

To run this section you need a free API key from <https://openrouter.ai>. Put it in
the environment variable `OPENROUTER_API_KEY`, or paste it inline (not recommended
for shared notebooks).

In [ ]:
import requests
import json

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")  # or paste your key

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# A few free model options. Try a different one if rate limits hit.
# The 'openrouter/free' router auto-picks a free model that fits the request.
FREE_MODEL = "openrouter/free"
# Alternatives you can swap in:
#   "meta-llama/llama-3.3-70b-instruct:free"
#   "mistralai/mistral-small-3.1:free"
#   "google/gemma-2-9b-it:free"
#   "qwen/qwen-2.5-7b-instruct:free"

def call_llm(messages, model=FREE_MODEL, temperature=0.0):
    if not OPENROUTER_API_KEY:
        raise RuntimeError("Set OPENROUTER_API_KEY before calling the API.")
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }
    r = requests.post(OPENROUTER_URL, headers=headers, data=json.dumps(payload), timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

# Quick smoke test
if OPENROUTER_API_KEY:
    out = call_llm([{"role": "user", "content": "Reply with the single word: ready."}])
    print("LLM said:", out)
else:
    print("No API key set; skipping live call. Set OPENROUTER_API_KEY to enable section 4.")

### Zero-shot classification

We ask the LLM to assign each sentence to one of three categories: *therapeutics*,
*epidemiology / risk factor*, or *statistical method*. No training data needed.

In [ ]:
CATEGORIES = ["therapeutics", "epidemiology", "statistical method"]

CLASSIFY_PROMPT = (
    "You are a careful biomedical text classifier.\n"
    "Read the sentence and assign it to exactly ONE of these categories: "
    f"{', '.join(CATEGORIES)}.\n"
    "Respond with only the category name, lowercase, no punctuation."
)

def classify(sentence):
    messages = [
        {"role": "system", "content": CLASSIFY_PROMPT},
        {"role": "user", "content": sentence},
    ]
    label = call_llm(messages).strip().lower()
    # robustness: keep only the matching category
    for c in CATEGORIES:
        if c in label:
            return c
    return label

if OPENROUTER_API_KEY:
    for s in corpus[:6]:
        print(f"[{classify(s):>20s}]  {s}")
else:
    print("Skipped (no API key).")

### LLM-based pairwise similarity

Instead of cosine similarity on embeddings, we can ask the LLM to score how
semantically related two sentences are. This is slower and not deterministic
across models, but it captures nuance that geometric similarity sometimes misses.

In [ ]:
SIMILARITY_PROMPT = (
    "You will be given two biomedical sentences. Rate how semantically similar "
    "they are on an integer scale from 0 (unrelated) to 5 (paraphrase). "
    "Respond with only the integer."
)

def llm_similarity(a, b):
    messages = [
        {"role": "system", "content": SIMILARITY_PROMPT},
        {"role": "user", "content": f"Sentence A: {a}\nSentence B: {b}"},
    ]
    out = call_llm(messages).strip()
    # extract first integer in the response
    digits = "".join(ch for ch in out if ch.isdigit())
    return int(digits[0]) if digits else None

pairs = [
    ("Aspirin reduces the risk of myocardial infarction.",
     "Acetylsalicylic acid lowers the chance of a heart attack."),
    ("Metformin is used to treat type 2 diabetes.",
     "Smoking is a risk factor for lung cancer."),
    ("Kaplan-Meier curves estimate the survival function.",
     "Cox regression models hazard rates over time."),
]

if OPENROUTER_API_KEY:
    for a, b in pairs:
        s = llm_similarity(a, b)
        print(f"score = {s}")
        print(f"   A: {a}")
        print(f"   B: {b}\n")
else:
    print("Skipped (no API key).")

### Getting actual embeddings from OpenRouter

OpenRouter also exposes an `/embeddings` endpoint that returns dense vectors,
identical in shape to what you would compute locally with `sentence-transformers`.
Most embedding models there are paid, but it is the same recipe shown below if
you swap to a paid model. The free chat models we used above do **not** support
this endpoint.

In [ ]:
EMBEDDINGS_URL = "https://openrouter.ai/api/v1/embeddings"

def get_embeddings(texts, model="qwen/qwen3-embedding-0.6b"):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {"model": model, "input": texts}
    r = requests.post(EMBEDDINGS_URL, headers=headers, data=json.dumps(payload), timeout=60)
    r.raise_for_status()
    data = r.json()["data"]
    return np.array([d["embedding"] for d in data])

# Example call (requires credits on OpenRouter; the embedding endpoint is generally not free):
# vecs = get_embeddings(corpus[:3])
# print(vecs.shape)
print("Function defined. Uncomment the call above if you have credits for an embedding model.")

## 5. Putting it all together

We have moved through three generations of text representation:

| Family | Captures | Pros | Cons |
|---|---|---|---|
| BoW / TF-IDF | word frequency | fast, sparse, interpretable | no semantics, no word order |
| Word2Vec / GloVe | word-level semantics from co-occurrence | dense, captures analogies | one vector per word, no context |
| Transformer sentence encoders | contextual semantics of full sentences | strong on similarity and retrieval | requires GPU for large models |
| LLMs via API | task-level reasoning over text | zero-shot capability, very flexible | costly, slower, less reproducible |

### When to use what (in a biostatistics workflow)

- For a quick logistic regression on clinical text labels: TF-IDF features are often
  a strong baseline.
- For information retrieval, deduplication, or clustering of PubMed abstracts:
  sentence-transformer embeddings.
- For zero-shot labeling of free-text fields in an EHR when you have no annotations:
  LLM prompting via OpenRouter or a similar API.
- For everything in between: a hybrid pipeline (embeddings for retrieval, LLM for
  reasoning on top of retrieved chunks - the standard RAG pattern).

### Exercises

1. Replace `glove-wiki-gigaword-50` with `glove-wiki-gigaword-300` and rerun the
   biomedical analogy. Does the answer change?
2. Use sentence embeddings to retrieve the top-3 most similar sentences in the
   corpus for the query *"how do statisticians model patient survival"*.
3. Build a confusion matrix between the LLM's zero-shot labels and your own labels
   for the 20 sentences in `corpus`. Where does the LLM disagree with you?
4. Pick a real PubMed abstract (200-500 words) and run all three representation
   families on it. Compare which method finds the closest neighbours in the corpus.
